# PCG Algorithms (overview)

A map of the **procedural content generation** (PCG) landscape — the families of algorithms
that *generate* game/sim content (terrain, levels, dungeons, names, textures, quests) instead
of an artist hand-authoring it. This notebook is the **index** for the rest of this domain: it
gives you the taxonomy, the shared vocabulary, and a runnable taste of three different families,
then points to the deep-dive notebook for each one.

**Domain:** Procedural Generation  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**PCG = creating content with an algorithm + a seed rather than by hand.** You write a generator
once; it produces an unbounded supply of levels, maps, items, or stories — each reproducible
from its seed, so you can store a whole world in a single integer.

**The problem it solves.** Hand-authored content doesn't scale. *No Man's Sky* ships 18 quintillion
planets; *Minecraft* streams an effectively infinite world; a roguelike must feel fresh every
run. None of that fits on a disk or in an art budget. PCG trades *storage and labour* for
*compute and a generator you can tune*.

**Reach for PCG when:**
- You need **volume or variety** beyond what humans can author — infinite worlds, endless levels,
  thousands of items/quests/names.
- You want **replayability** — a new layout every run (roguelikes, daily challenges).
- You want **adaptivity** — content shaped to the player, the hardware, or runtime constraints.
- You want **compression** — regenerate from a seed instead of storing the artifact.
- You're **prototyping** — fill a world cheaply before artists touch it.

**Look elsewhere when:**
- The content is **narratively load-bearing** — a hand-crafted boss, a scripted story beat. PCG
  excels at *texture and breadth*, struggles with *intentional, meaningful* structure.
- You need **guaranteed quality/solvability** and can't afford a validation pass — generators
  produce duds; you must test or constrain the output.
- A few fixed assets suffice — a generator is overkill (and a maintenance liability) for content
  you'd author once anyway.

The classic warning is Kate Compton's **"10,000 bowls of oatmeal"**: it's easy to make 10,000
*technically unique* bowls of oatmeal that all look the same to a player. **Perceptual** variety,
not just numerical variety, is the real goal.

## 2. Mental Model

**A PCG generator is a function `content = generate(seed, params)` wrapped in an optional
generate-and-test loop.**

```
        params (knobs: size, density, difficulty, style)
           │
  seed ───▶│  GENERATOR  │───▶ candidate content
           └─────────────┘            │
                                       ▼
                                 ┌───────────┐   fail
                                 │   TEST    │────────┐
                                 │ valid? fun?│       │ reject,
                                 │ solvable? │        │ re-seed
                                 └───────────┘        │
                                       │ pass         │
                                       ▼              │
                                  ship it  ◀──────────┘
```

Two dimensions organize the whole field:

- **Constructive vs. generate-and-test vs. search-based.** *Constructive* builds valid content
  in one pass by construction (grammars, agent walks). *Generate-and-test* generates then
  filters (the loop above). *Search-based* puts a fitness function in the loop and *optimizes*
  (evolutionary algorithms, WFC's constraint propagation).
- **What the randomness drives.** *Noise/fractal* methods randomize a continuous field;
  *grammar/symbolic* methods randomize rule choices; *agent/simulation* methods randomize a
  walker's decisions; *constraint/solver* methods randomize within a satisfiability search;
  *ML* methods sample a learned distribution.

Hold those two axes in your head and any PCG technique slots into place.

## 3. Key Concepts

- **Seed** — the integer (RNG state) that makes generation **deterministic and reproducible**.
  Same seed + same code + same params ⇒ identical content. This is what lets you store a world
  as one number and reproduce bug reports.
- **Generator** — the `seed, params → content` function. Its **content space** is the set of all
  outputs it can ever produce.
- **Expressivity / expressive range** — how varied and how *biased* that content space is. A
  generator can be unique-per-seed yet perceptually monotonous (the oatmeal problem). Plot the
  expressive range to see what it actually covers.
- **Controllability** — how predictably `params` steer the output (a "difficulty" knob, a
  "density" knob). More control usually means less surprise; it's a genuine trade-off.
- **Constructive vs. generate-and-test vs. search-based** — the three control strategies from the
  mental model.
- **Online vs. offline** — *online* generates during play (must be fast and always valid:
  *Minecraft* chunks); *offline* generates ahead of time and can afford expensive search/curation.
- **Validity / playability constraint** — the predicate the output must satisfy (connected,
  solvable, reachable exit, no soft-locks). Enforced by construction, by rejection, or by repair.
- **Constraint solving** — generation framed as "fill the grid so all local rules hold"
  (Wave Function Collapse, answer-set programming).
- **Mixed-initiative / authored hooks** — a human guides or seeds the generator; PCG fills the
  rest. Often the sweet spot for narrative content.
- **The generator families** (each has its own notebook in this domain):
  | Family | Idea | Notebook |
  |---|---|---|
  | Noise / fractal | smooth random fields → heightmaps, textures | `perlin-noise`, `noise-simplex-worley`, `diamond-square` |
  | Point distributions | even-but-random placement / partitions | `poisson-disk-sampling`, `voronoi-delaunay` |
  | Agent / simulation | a walker carves space step by step | `drunkards-walk`, `bsp-dungeon-generation` |
  | Cellular automata | iterate local rules over a grid | `cellular-automata` |
  | Grammar / symbolic | rewrite rules expand symbols → structure/text | `l-systems`, `context-free-grammars`, `tracery`, `markov-chains` |
  | Constraint / solver | satisfy local adjacency constraints | `wave-function-collapse` |
  | Rule-based | designer rules + tables assemble content | `rule-based-generation` |
  | Machine learning | sample a learned distribution | `gan-diffusion-pcg` |

## 4. Setup

Everything here runs on **NumPy alone** — CPU-only, no GPU, no network, no API key. The point
is to *see the families side by side* in a few lines each; the per-family notebooks go deeper.

```bash
%pip install numpy
```

The deep-dive notebooks occasionally reach for extras (`matplotlib` to render images, `noise`
or `opensimplex` for fast C noise, `scipy` for spatial structures, ML stacks for
`gan-diffusion-pcg`). None are needed for this overview.

In [ ]:
import numpy as np

print("NumPy", np.__version__)

# The whole of PCG starts here: a *seeded* random source. Reproducibility — same
# seed, same content — is the property every technique below is built on.
def show_seed(seed):
    rng = np.random.default_rng(seed)
    return rng.integers(0, 100, size=5)

print("seed=7  ->", show_seed(7))
print("seed=7  ->", show_seed(7), "(identical: a seed *is* the content)")
print("seed=8  ->", show_seed(8), "(different seed, different content)")

## 5. Worked Examples

Three examples, building up the core PCG idea:

1. **Seed + generate-and-test** — the loop that underlies most practical PCG.
2. **Three generator families on one canvas** — noise, agent-based, and cellular automata,
   so the taxonomy is concrete, not abstract.
3. **Grammar-based generation** — the symbolic family, producing text instead of grids.

### Example 1 — Seeded determinism + the generate-and-test loop

The bread-and-butter PCG pattern: generate a candidate, **test it against a constraint**, and
re-seed until one passes (*rejection sampling*). Here a **drunkard's walk** (a random walker that
carves floor) makes a cave; we demand it carve at least 41% of the map. Note the cost: a tight
constraint can reject many candidates — a real tension between *control* and *generation effort*
(`drunkards-walk` covers this generator in depth).

In [ ]:
def drunkards_walk(shape, steps, seed):
    """A walker starts in the middle and stumbles around, carving floor (1) into wall (0)."""
    rng = np.random.default_rng(seed)
    grid = np.zeros(shape, dtype=int)
    y, x = shape[0] // 2, shape[1] // 2
    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    for _ in range(steps):
        grid[y, x] = 1
        dy, dx = moves[rng.integers(4)]
        y = int(np.clip(y + dy, 0, shape[0] - 1))
        x = int(np.clip(x + dx, 0, shape[1] - 1))
    return grid

# Generate-and-test: keep re-seeding until the carved area meets the constraint.
target = 0.41                      # require >= 41% floor — a deliberately tight ask
shape, steps = (18, 48), 1100
for seed in range(10_000):
    cave = drunkards_walk(shape, steps, seed)
    coverage = cave.mean()
    if coverage >= target:
        break

print(f"accepted seed={seed} after rejecting {seed} candidate(s); "
      f"floor coverage={coverage:.1%}")

# Determinism: the accepted seed reproduces the exact same cave, every time.
again = drunkards_walk(shape, steps, seed)
print("re-running the accepted seed is identical:", np.array_equal(cave, again))

print("\n".join("".join(" " if v else "#" for v in row) for row in cave))

### Example 2 — Three generator families on one canvas

The same `(14 x 48)` grid, generated three different ways. Read these as **representatives of
whole families**, not just three functions:

- **Noise / fractal** — randomize a *continuous field*, then threshold it. Organic, blobby
  (`perlin-noise`, `noise-simplex-worley`).
- **Agent / simulation** — a *walker* makes local decisions and carves as it goes. Connected,
  wandering corridors (`drunkards-walk`, `bsp-dungeon-generation`).
- **Cellular automata** — start from random noise, then *iterate a local rule* (a cell becomes
  wall if ≥5 of its 8 neighbours are walls). Random soup smooths into rounded caverns
  (`cellular-automata`).

In [ ]:
def value_noise(shape, cells, seed):
    """Smooth random field: random values on a coarse lattice, smoothstep-interpolated up."""
    rng = np.random.default_rng(seed)
    coarse = rng.random((cells + 1, cells + 1))
    ys = np.linspace(0, cells, shape[0]); xs = np.linspace(0, cells, shape[1])
    y0 = np.floor(ys).astype(int); x0 = np.floor(xs).astype(int)
    y1 = np.minimum(y0 + 1, cells); x1 = np.minimum(x0 + 1, cells)
    fy = (ys - y0)[:, None]; fx = (xs - x0)[None, :]
    fy = fy * fy * (3 - 2 * fy); fx = fx * fx * (3 - 2 * fx)   # smoothstep ease
    c00 = coarse[np.ix_(y0, x0)]; c10 = coarse[np.ix_(y0, x1)]
    c01 = coarse[np.ix_(y1, x0)]; c11 = coarse[np.ix_(y1, x1)]
    return (c00 * (1 - fx) + c10 * fx) * (1 - fy) + (c01 * (1 - fx) + c11 * fx) * fy

def ca_cave(shape, fill, iters, seed):
    """Cellular automata: random walls, then smooth with the classic 4-5 rule."""
    rng = np.random.default_rng(seed)
    grid = (rng.random(shape) < fill).astype(int)            # 1 = wall
    for _ in range(iters):
        p = np.pad(grid, 1, constant_values=1)               # treat the border as wall
        walls = sum(p[i:i + shape[0], j:j + shape[1]]
                    for i in range(3) for j in range(3)) - grid
        grid = (walls >= 5).astype(int)
    return grid

def render(mask, title):
    print(title)
    print("\n".join("".join(" " if v else "#" for v in row) for row in mask))
    print()

H, W = 14, 48
noise = value_noise((H, W), cells=6, seed=2)
render((noise > noise.mean()).astype(int), "noise / fractal  (value noise, thresholded):")
render(drunkards_walk((H, W), 700, seed=2), "agent / simulation  (drunkard's walk):")
render(1 - ca_cave((H, W), fill=0.45, iters=4, seed=2), "cellular automata  (4-5 smoothed cave):")

### Example 3 — Grammar-based generation (the symbolic family)

Not all PCG is grids. **Grammars** randomize *rule choices* to expand a start symbol into
structure — the basis of L-systems (plants), Tracery (bots/flavor text), and context-free
grammars (`l-systems`, `tracery`, `context-free-grammars`). Here a tiny recursive grammar
generates fantasy place names; the same seed gives the same names, so the whole batch is one
reproducible integer.

In [ ]:
import re

grammar = {
    "name":   ["{prefix}{suffix}", "{prefix}{vowel}{suffix}"],
    "prefix": ["Mor", "Thar", "El", "Gond", "Kaz", "Bryn", "Vor"],
    "vowel":  ["a", "o", "ia", "en", "ur"],
    "suffix": ["dor", "heim", "wyn", "gard", "fell", "moor", "thas"],
}

def expand(grammar, symbol, rng):
    """Recursively expand a symbol; pick a random production and fill its {tokens}."""
    rule = grammar.get(symbol)
    if rule is None:
        return symbol                                  # terminal: a literal string
    production = rule[rng.integers(len(rule))]         # randomized rule choice
    return re.sub(r"\{(\w+)\}",
                  lambda m: expand(grammar, m.group(1), rng), production)

rng = np.random.default_rng(42)
names = [expand(grammar, "name", rng) for _ in range(10)]
print("seed=42 ->", ", ".join(names))

# Reproducible: a fresh RNG with the same seed yields the same batch.
rng2 = np.random.default_rng(42)
print("again   ->", ", ".join(expand(grammar, "name", rng2) for _ in range(10)))

## 6. Gotchas & Pitfalls

- **The oatmeal problem (perceptual vs numerical variety).** "10,000 unique levels" is
  meaningless if they all *feel* the same. Design for variety the player can *perceive* —
  vary structure and pacing, not just pixel-level seeds. Plot the **expressive range** to check.
- **Generators produce duds.** Unconstrained PCG happily emits unsolvable mazes, disconnected
  caves, or impossible difficulty spikes. **Always** enforce validity — by construction, by
  generate-and-test (Example 1), or by a repair pass. Never ship unvalidated generated content.
- **Rejection sampling can be expensive.** Tight constraints reject many candidates (Example 1
  rejected dozens). If acceptance is rare, switch from "generate-and-test" to "generate *valid by
  construction*" or to a constraint solver (WFC) that never makes invalid moves.
- **Seed reproducibility is fragile across environments.** Determinism holds only with the *same*
  RNG, library version, and platform. Python's `hash()` is salted per process; float math differs
  across hardware. Use an explicit seeded RNG (`np.random.default_rng`), avoid `hash()` for
  seeding, and pin versions if seeds must travel.
- **Online generators must be fast *and* always valid.** Content generated during play (streamed
  chunks) can't pause for a slow search or reject-and-retry — design it to be valid by
  construction. Offline generation can afford expensive curation.
- **Controllability vs. surprise.** Bolt on enough knobs and constraints and the output becomes
  predictable and dull; too few and it's uncontrollable. Tune deliberately, and expose only the
  knobs that matter.
- **Debugging stochastic systems.** "It broke sometimes" is agony without the seed. **Log the
  seed with every artifact** so any output (and any bug) is reproducible.
- **ML generators need data and can memorize.** GAN/diffusion PCG (`gan-diffusion-pcg`) requires a
  corpus of examples and can overfit (regurgitate training levels) or produce *plausible-but-broken*
  output that still needs a validity check. It is not a shortcut around the dud problem.

## 7. When to Use vs Alternatives

Pick a family by **what you're generating** and **how much control/validity you need**:

| Family | Best for | Watch out for | Deep dive |
|---|---|---|---|
| **Noise / fractal** | Continuous fields: terrain, textures, clouds, biomes. | Not discrete structure; tiling/artifacts. | `perlin-noise`, `noise-simplex-worley`, `diamond-square` |
| **Point distributions** | Even-random placement, region maps, partitions. | Produces points/cells, not whole levels. | `poisson-disk-sampling`, `voronoi-delaunay` |
| **Agent / simulation** | Connected caves, corridors, organic dungeons. | Hard to constrain global shape; can wander. | `drunkards-walk`, `bsp-dungeon-generation` |
| **Cellular automata** | Organic caverns from noise; cheap, tunable. | Needs post-processing for connectivity. | `cellular-automata` |
| **Grammar / symbolic** | Text, names, plants, hierarchical structure. | Authoring the grammar is the real work. | `l-systems`, `context-free-grammars`, `tracery`, `markov-chains` |
| **Constraint / solver (WFC)** | Tile maps that must satisfy local adjacency rules. | Can fail/contradict; slower; needs a tileset. | `wave-function-collapse` |
| **Rule-based** | Designer-authored tables/rules (loot, encounters). | As good as the rules; limited emergent surprise. | `rule-based-generation` |
| **Search / evolutionary** | Optimizing content to a fitness function. | Needs a good fitness fn; expensive; offline. | (search-based PCG, see Resources) |
| **Machine learning** | Mimicking a corpus of human-made content. | Needs data; overfits; outputs still need validation. | `gan-diffusion-pcg` |

**Versus the non-PCG alternatives:**
- **Hand-authoring** wins when content is narratively load-bearing or small in volume. PCG wins on
  scale, replayability, and compression. **Mixed-initiative** (human seeds, generator fills) often
  beats either alone.
- **Combine families.** Real generators stack them: noise for the heightmap, Voronoi for biomes,
  agents/BSP for the dungeon, grammars for quests and names, a validity pass over all of it.

**Rule of thumb:** start from the *simplest* family that can express your content
(noise or an agent walk), add a **generate-and-test** validity loop, and only escalate to
constraint solvers or ML when simpler methods can't hit your variety or quality bar.

## 8. Resources

- **Procedural Content Generation in Games (Shaker, Togelius & Nelson) — the free textbook** —
  http://pcgbook.com/
- **Togelius et al., "Search-Based Procedural Content Generation: A Taxonomy and Survey" (IEEE)** —
  http://julian.togelius.com/Togelius2011Searchbased.pdf
- **Kate Compton — "So you want to build a generator…" (the 10,000 bowls of oatmeal essay)** —
  https://www.galaxykate.com/blog/generator-design.html
- **Herbert Wolverson — "Procedural Map Generation Techniques" / the Roguelike Tutorial** —
  https://bfnightly.bracketproductions.com/chapter_23.html
- **Red Blob Games — interactive guides to maps, noise, and grids** —
  https://www.redblobgames.com/
- **Maxim Gumin — Wave Function Collapse (reference implementation & gallery)** —
  https://github.com/mxgmn/WaveFunctionCollapse
- **PROCJAM — the procedural generation jam, with a free tutorials archive** —
  https://www.procjam.com/
- Deep-dive notebooks in this domain: `perlin-noise`, `noise-simplex-worley`, `diamond-square`,
  `poisson-disk-sampling`, `voronoi-delaunay`, `drunkards-walk`, `bsp-dungeon-generation`,
  `cellular-automata`, `l-systems`, `context-free-grammars`, `tracery`, `markov-chains`,
  `wave-function-collapse`, `rule-based-generation`, `gan-diffusion-pcg`.

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def generate_and_test(seed, generate, validate, attempts=100):
    ...


def expressive_range(seeds, generate, measures):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE